In [0]:
%run ../config_setup

# Silver 02 — Site (SCD2)

RDS CDC se `site` events uthata hai aur env-configured `site` silver table me PySpark SCD2 apply karta hai (latest `_lsn` per site).

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, get_json_object, current_timestamp, expr, lit
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()
TARGET_TABLE = f"{SILVER}.site"

df = spark.readStream.table(f"{BRONZE}.rds_cdc").filter(col("table") == "site").select(
    col("_lsn"), col("_change_type"),
    get_json_object("after", "$.site_id").alias("site_id"),
    get_json_object("after", "$.site_name").alias("site_name"),
    get_json_object("after", "$.city").alias("city"),
    get_json_object("after", "$.country_code").alias("country_code"),
    get_json_object("after", "$.geo_lat").cast("double").alias("geo_lat"),
    get_json_object("after", "$.geo_lon").cast("double").alias("geo_lon")
)

def upsert_site(microBatchDF, batchId):
    if microBatchDF.count() == 0: return
    latest = microBatchDF.withColumn("rn", expr("row_number() over (partition by site_id order by _lsn desc)")).filter("rn = 1").drop("rn")

    if not spark.catalog.tableExists(TARGET_TABLE):
        (latest.filter(col("_change_type") != "delete").drop("_lsn", "_change_type")
               .withColumn("__START_AT", current_timestamp()).withColumn("__END_AT", lit(None).cast("timestamp")).withColumn("__CURRENT", lit(True))
               .write.format("delta").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE))
    else:
        target = DeltaTable.forName(spark, TARGET_TABLE)
        target.alias("t").merge(latest.filter(col("_change_type") == "delete").alias("s"), "t.site_id = s.site_id AND t.__CURRENT = true").whenMatchedUpdate(set={"__END_AT": current_timestamp(), "__CURRENT": lit(False)}).execute()
        upserts = latest.filter(col("_change_type") != "delete")
        target.alias("t").merge(upserts.alias("s"), "t.site_id = s.site_id AND t.__CURRENT = true").whenMatchedUpdate(set={"__END_AT": current_timestamp(), "__CURRENT": lit(False)}).execute()
        (upserts.drop("_lsn", "_change_type").withColumn("__START_AT", current_timestamp()).withColumn("__END_AT", lit(None).cast("timestamp")).withColumn("__CURRENT", lit(True))
                .write.format("delta").mode("append").saveAsTable(TARGET_TABLE))

df.writeStream.foreachBatch(upsert_site).option("checkpointLocation", f"{CHECKPOINTS}/silver_site").trigger(availableNow=True).start().awaitTermination()